# CropIQ Phase 5: What-If Crop Yield Scenario Simulator

**Subtitle:** AI-Powered Crop Yield Intelligence  
**Tagline:** Predict. Understand. Optimize.  
**Objective:** Build an interactive What-If Scenario Simulator allowing farmers and agronomists to explore how the trained model's yield estimates respond to hypothetical changes in soil moisture, rainfall, temperature, and vegetation health.

```text
Current Farm Conditions
        ↓
Current Model Prediction
        ↓
User Modifies Selected Variable
        ↓
Scenario Input (Deep Copied)
        ↓
SAME CropIQ ML Model
        ↓
Scenario Prediction & Delta Explanation
        ↓
Compare Current vs Scenario (MAE Materiality)
        ↓
Non-Causal Insights & Caveats
```

## 1. Imports & Environment Setup
Import ML model utilities, intelligence layers, recommendation engine, and scenario simulator modules.

In [ ]:
import sys
import json
from pathlib import Path
import pandas as pd
import numpy as np

# Ensure project root is in sys.path
for p in [Path.cwd(), Path.cwd().parent]:
    if (p / 'src').exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

# CropIQ Core Modules
from src.ml.predict import load_prediction_model
from src.intelligence import analyze_crop_prediction
from src.recommendations import generate_recommendations
from src.simulator import (
    simulate_scenario,
    simulate_multiple_scenarios,
    compute_feature_sensitivity,
    create_preset_scenario,
    reset_scenario,
    load_scenario_metadata,
    compare_scenarios,
    session_history,
)

print('CropIQ Phase 5 modules successfully imported.')

CropIQ Phase 5 modules successfully imported.


## 2. Load Trained CropIQ ML Model
Load and verify the Phase 2 Random Forest regression pipeline (`models/cropiq_yield_model.joblib`).

In [ ]:
pipe = load_prediction_model()
print('Loaded Pipeline Steps:', list(pipe.named_steps.keys()))
print('Model Class:', type(pipe.named_steps['model']).__name__)
print('Number of Decision Trees:', pipe.named_steps['model'].n_estimators)

Loaded Pipeline Steps: ['preprocessor', 'model']
Model Class: RandomForestRegressor
Number of Decision Trees: 300


## 3. Load Scenario Metadata & Training Quantiles
Inspect modifiability classifications, physical limits, and empirical training percentiles from `knowledge/scenario_features.json`.

In [ ]:
meta = load_scenario_metadata()
features_meta = meta['features']
print(f'Registered Scenario Features: {list(features_meta.keys())}')
print(f'Validation MAE Materiality Threshold: {meta.get("material_change_threshold")} unconfirmed')

moist_stats = features_meta['soil_moisture']['training_stats']
print(f'Soil Moisture Empirical Bounds: Min={moist_stats["min"]}, Median={moist_stats["median"]}, Max={moist_stats["max"]}')

Registered Scenario Features: ['soil_moisture', 'rainfall', 'temperature', 'NDVI', 'GNDVI', 'NDWI', 'SAVI']
Validation MAE Materiality Threshold: 0.9765 unconfirmed
Soil Moisture Empirical Bounds: Min=4.789, Median=27.2845, Max=102.1287


## 4. Verify Phase 3 Intelligence Layer Connection
Confirm availability of local SHAP/ablation explanations, 4-factor risk scoring, and ensemble uncertainty.

In [ ]:
print('Phase 3 Intelligence Layer is ready to compute delta explanations and risk progressions.')

Phase 3 Intelligence Layer is ready to compute delta explanations and risk progressions.


## 5. Connect Phase 4 Recommendation Triggers
Demonstrate the product chain: Actionable Recommendation $\to$ Interactive What-If Simulation.

In [ ]:
from src.recommendations import load_agricultural_rules
rules = load_agricultural_rules()
simulatable_rules = [r for r in rules if r.get('what_if_supported')]
print(f'Total Rules with What-If Simulation Support: {len(simulatable_rules)} / {len(rules)}')
for r in simulatable_rules[:3]:
    print(f" - [{r['id']}] {r['title']} (Feature: {r['required_features']})")

Total Rules with What-If Simulation Support: 6 / 19
 - [water_001] Review soil moisture and irrigation needs (Feature: ['soil_moisture'])
 - [water_002] Maintain adequate moisture and avoid over-irrigation (Feature: ['soil_moisture'])
 - [water_003] Mitigate combined drought and heat moisture stress (Feature: ['soil_moisture', 'rainfall', 'temperature'])


## 6. Define Example Farm Observation
Load a realistic test observation from `data/processed/crop_yield_model_data.csv`.

In [ ]:
data_path = Path('data/processed/crop_yield_model_data.csv')
if not data_path.exists():
    data_path = Path('../data/processed/crop_yield_model_data.csv')

df = pd.read_csv(data_path)
sample_input = df.iloc[0].to_dict()
sample_input.pop('yield', None)

print('Baseline Crop:', sample_input['crop_type'])
print('Baseline Soil Moisture:', sample_input['soil_moisture'])
print('Baseline Rainfall:', sample_input['rainfall'])
print('Baseline Temperature:', sample_input['temperature'])

Baseline Crop: Rice
Baseline Soil Moisture: 46.11935326
Baseline Rainfall: 1.662354196
Baseline Temperature: 9.884228735


## 7. Compute Baseline Prediction & Intelligence
Establish the baseline yield, risk score, and tree ensemble uncertainty before applying any scenario changes.

In [ ]:
base_intel = analyze_crop_prediction(sample_input, pipeline=pipe)
base_yield = base_intel['prediction']['yield']
base_risk = base_intel['risk']['level']
base_unc = base_intel['uncertainty']['classification']

print(f'Baseline Predicted Yield: {base_yield:.2f} {base_intel["prediction"]["unit"]}')
print(f'Baseline Risk: {base_risk} (Score: {base_intel["risk"]["score"]}/100)')
print(f'Baseline Uncertainty: {base_unc}')

Baseline Predicted Yield: 36.43 unconfirmed
Baseline Risk: MODERATE (Score: 59/100)
Baseline Uncertainty: HIGH


## 8. Single-Variable Scenario: Soil Moisture Adjustment
Simulate moving root-zone moisture from current baseline to an elevated range ($34.0$ unconfirmed).

In [ ]:
single_res = simulate_scenario(
    current_input=sample_input,
    scenario_changes={'soil_moisture': 34.0},
    scenario_name='Increased Soil Moisture',
    pipeline=pipe,
)
print('Scenario Yield:', single_res['scenario']['predicted_yield'])
print('Absolute Difference:', single_res['comparison']['absolute_change'])
print('Direction:', single_res['comparison']['direction'])
print('Materiality:', single_res['comparison']['materiality_label'])

Scenario Yield: 36.4639
Absolute Difference: 0.0378
Direction: no_material_change
Materiality: small model-estimated difference


## 9. Multi-Variable Scenario: Combined Weather & Soil Changes
Simulate simultaneous changes in soil moisture, rainfall, and temperature.

In [ ]:
multi_res = simulate_scenario(
    current_input=sample_input,
    scenario_changes={'soil_moisture': 32.0, 'rainfall': 14.0, 'temperature': 22.0},
    scenario_name='Cooler & Moist Conditions',
    pipeline=pipe,
)
print('Scenario Yield:', multi_res['scenario']['predicted_yield'])
print('Absolute Difference:', multi_res['comparison']['absolute_change'])
print('Warnings:', multi_res['validation']['warnings'])

Scenario Yield: 44.5099
Absolute Difference: 8.0838
Warnings: ["Note: 'rainfall' is an external contextual variable; this scenario simulates hypothetical environmental conditions, not a farmer-controlled action.", "Note: 'temperature' is an external contextual variable; this scenario simulates hypothetical environmental conditions, not a farmer-controlled action.", 'Because multiple inputs were changed together, the estimated difference cannot be attributed to one variable alone; joint interactions may influence the model.']


## 10. Scenario Comparison & Feature Diff Table
Review structured comparison metrics, percentage changes, and feature diffs.

In [ ]:
diff_rows = []
for f, d in multi_res['comparison']['feature_diffs'].items():
    diff_rows.append({
        'Feature': f,
        'Baseline': d['baseline'],
        'Scenario': d['scenario'],
        'Difference': d['difference'],
    })
diff_df = pd.DataFrame(diff_rows)
print(diff_df.to_string(index=False))

      Feature  Baseline  Scenario  Difference
soil_moisture   46.1194      32.0    -14.1194
     rainfall    1.6624      14.0     12.3376
  temperature    9.8842      22.0     12.1158


## 11. Scenario Explainability & Delta Contributions
Identify which feature contributions changed between baseline and scenario to account for the model estimate difference.

In [ ]:
delta_contribs = multi_res['explanation']['delta_contributors']
contrib_df = pd.DataFrame(delta_contribs)
print(contrib_df.to_string(index=False))

      feature  baseline_contribution  scenario_contribution  delta_contribution      direction
     rainfall                -1.4695                 7.6784              9.1479 positive_shift
  temperature                 0.1182                -0.0083             -0.1265 negative_shift
soil_moisture                -0.0837                -0.1353             -0.0516 negative_shift


## 12. Scenario Risk Progression Tracking
Compare baseline crop yield risk with scenario risk.

In [ ]:
risk_prog = multi_res['progression']['risk']
print('Baseline Risk:', risk_prog['baseline_level'], f"({risk_prog['baseline_score']}/100)")
print('Scenario Risk:', risk_prog['scenario_level'], f"({risk_prog['scenario_score']}/100)")
print('Risk Classification Changed:', risk_prog['risk_changed'])

Baseline Risk: MODERATE (59/100)
Scenario Risk: LOW (16/100)
Risk Classification Changed: True


## 13. Scenario Uncertainty Progression
Evaluate decision tree spread under the modified scenario conditions.

In [ ]:
unc_prog = multi_res['progression']['uncertainty']
print('Baseline Uncertainty:', unc_prog['baseline_level'])
print('Scenario Uncertainty:', unc_prog['scenario_level'])
print('Relative Uncertainty Spread:', f"{unc_prog['relative_uncertainty']*100:.1f}%")

Baseline Uncertainty: HIGH
Scenario Uncertainty: LOW
Relative Uncertainty Spread: 10.1%


## 14. Out-of-Distribution Scenario Test
Test scenario behavior when an input exceeds historical training extremes.

In [ ]:
ood_res = simulate_scenario(
    current_input=sample_input,
    scenario_changes={'rainfall': 110.0},  # Training max is 93.36
    scenario_name='Extreme Rainfall Event',
    pipeline=pipe,
)
print('Within Training Range:', ood_res['validation']['within_training_range'])
print('Scenario Reliability:', ood_res['interpretation']['scenario_reliability'])
print('Reliability Rationale:', ood_res['interpretation']['reliability_reason'])
print('OOD Warnings:', [w for w in ood_res['validation']['warnings'] if 'outside the model' in w])

Within Training Range: False
Scenario Reliability: LOW
Reliability Rationale: Modified inputs require model extrapolation beyond observed training extremes.
OOD Warnings: ["Scenario value for 'rainfall' (110.0) is outside the model's observed training range [0.87, 93.37]. The model estimate requires extrapolation and may be less reliable."]


## 15. 1D Feature Sensitivity Analysis Curve
Generate model sensitivity curve for soil moisture holding all other features fixed.

In [ ]:
sens_curve = compute_feature_sensitivity(
    current_input=sample_input,
    feature_name='soil_moisture',
    num_points=8,
    pipeline=pipe,
)
points_df = pd.DataFrame(sens_curve['points'])
print(f"Sensitivity curve for {sens_curve['display_name']} ({len(points_df)} points):")
print(points_df.to_string(index=False))
print('\nDisclaimer:', sens_curve['disclaimer'])

Sensitivity curve for Soil Moisture (8 points):
 feature_value  predicted_yield  difference
        6.4552          36.5807      0.1546
       16.6700          36.4646      0.0385
       26.8848          36.3933     -0.0328
       37.0996          36.4639      0.0378
       47.3143          36.4238     -0.0023
       57.5291          36.5120      0.0859
       67.7439          36.5267      0.1006
       77.9587          36.5267      0.1006

Disclaimer: This curve depicts model sensitivity for soil_moisture holding other features constant. It represents learned statistical associations, not a physical crop response curve.


## 16. Multiple Scenario Batch Comparison (Empirical Presets)
Simulate and rank a batch of empirical quantile presets.

In [ ]:
preset_scenarios = [
    create_preset_scenario(sample_input, 'improve_moisture'),
    create_preset_scenario(sample_input, 'historical_median_moisture'),
    create_preset_scenario(sample_input, 'drought_stress'),
    create_preset_scenario(sample_input, 'higher_rainfall'),
    create_preset_scenario(sample_input, 'cooler_temperature'),
]
batch_res = simulate_multiple_scenarios(sample_input, preset_scenarios, pipeline=pipe)
print('Comparative Summary:', batch_res['comparative_summary'])

ranked_df = pd.DataFrame(batch_res['scenarios_ranked'])
print(ranked_df[['name', 'scenario_yield', 'difference', 'direction', 'risk_level', 'reliability']].to_string(index=False))

Comparative Summary: Among the 5 tested scenarios, 'Higher Precipitation Scenario' produced the highest model-estimated yield. This comparison reflects simulated model associations and is not a guaranteed agronomic optimum.
                               name  scenario_yield  difference          direction risk_level reliability
      Higher Precipitation Scenario         42.2007      5.7746           increase        LOW        HIGH
   Drought Moisture Stress Scenario         36.5807      0.1546 no_material_change   MODERATE      MEDIUM
        Favorable Moisture Scenario         36.4639      0.0378 no_material_change   MODERATE      MEDIUM
        Cooler Temperature Scenario         36.4340      0.0079 no_material_change   MODERATE      MEDIUM
Historical Median Moisture Scenario         36.3933     -0.0328 no_material_change   MODERATE      MEDIUM


## 17. Visualization Data Preparation
Format comparative chart data ready for frontend visualization (Phase 7).

In [ ]:
chart_data = [
    {'Scenario': 'Baseline', 'Estimated Yield': base_yield}
]
for row in batch_res['scenarios_ranked']:
    chart_data.append({'Scenario': row['name'], 'Estimated Yield': row['scenario_yield']})

chart_df = pd.DataFrame(chart_data)
print('Frontend Bar Chart Payload:')
print(chart_df.to_string(index=False))

Frontend Bar Chart Payload:
                           Scenario  Estimated Yield
                           Baseline          36.4261
      Higher Precipitation Scenario          42.2007
   Drought Moisture Stress Scenario          36.5807
        Favorable Moisture Scenario          36.4639
        Cooler Temperature Scenario          36.4340
Historical Median Moisture Scenario          36.3933


## 18. Output Contract Validation & Safety Disclaimers
Verify complete structured JSON schema, non-causal claims flag, and reset capability.

In [ ]:
# 1. Verify causal claim flag is strictly False
assert single_res['interpretation']['causal_claim'] is False, 'Causal claim must be strictly False!'

# 2. Verify summary text includes mandatory disclaimer
assert 'model-based scenario estimate' in single_res['interpretation']['summary']

# 3. Test Reset Functionality
reset_res = reset_scenario(sample_input)
assert reset_res['status'] == 'reset'
assert len(reset_res['changes']) == 0

print('All Phase 5 schema contracts, safety flags, and reset capabilities validated successfully.')

All Phase 5 schema contracts, safety flags, and reset capabilities validated successfully.
